In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import vectorbt as vbt
import sys

sys.path.insert(0, str(Path('.').resolve().parent))
from dema_strat import load_ohlcv, vbt_frame, free, build_chandelier_exits

CASH = 50_000
COMMISSION = 0.00015
TABLE_NAME = "f0_smoothing_chandelier_exit"
SMOOTH_PERIOD = 12

def load_data(file_path, *, verbose=False, use_vwap=False):
    # Compact float32 + Parquet-cached loader.
    return load_ohlcv(file_path, use_cache=True, add_vwap=use_vwap, verbose=verbose)

def make_signals(data, n1=9, atr_period=14, trail_mult=2.0, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    # keep float32 through indicator math; cast up only at the vectorbt boundary
    close = data["Close"]
    high = data["High"]
    low = data["Low"]

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    smoothed = vbt.indicators.MA.run(ema1.ma, window=smooth_period, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = trail_mult * atr.atr
    sl_pct = sl_dist / close
    long_mask = (smoothed.ma > ema1.ma) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_mask = (ema1.ma > smoothed.ma) & np.isfinite(atr.atr) & (atr.atr > 0)

    if use_vwap:
        long_mask = long_mask & (close > data["VWAP"])
        short_mask = short_mask & (close < data["VWAP"])
    long_signal = long_mask
    short_signal = short_mask

    return pd.DataFrame({
        "ema1": ema1.ma,
        "smoothed": smoothed.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "long": long_signal,
        "short": short_signal,
    })

def run_backtest(data, n1=9, atr_period=14, trail_mult=2.0, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    signals = make_signals(data, n1=n1, atr_period=atr_period, trail_mult=trail_mult, use_vwap=use_vwap, smooth_period=smooth_period)
    freq = data.index.to_series().diff().median()

    long_signal  = signals["long"].to_numpy()
    short_signal = signals["short"].to_numpy()
    # vbt wants float64 for size/close; cast just the columns it needs.
    close_np = data["Close"].to_numpy(dtype=np.float64)

    long_exits, short_exits = build_chandelier_exits(data, signals, trail_mult)

    portfolio = vbt.Portfolio.from_signals(
        close=close_np,
        entries=long_signal,
        exits=long_exits,
        short_entries=short_signal,
        short_exits=short_exits,
        init_cash=CASH,
        fees=COMMISSION,
        size=100.0,
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(dtype=np.float64),
        tp_stop=None,
    )

    return portfolio, portfolio.stats(), signals

def run_params(data, n1, atr_period, trail_mult, use_vwap=False, asset=None, timeframe=None, smooth_period=SMOOTH_PERIOD):
    try:
        portfolio, stats, _ = run_backtest(data, n1=n1, atr_period=atr_period, trail_mult=trail_mult, use_vwap=use_vwap, smooth_period=smooth_period)
        return {
            "n1": n1, "atr_period": atr_period,
            "trail_mult": trail_mult,
            "use_vwap": use_vwap,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "return_pct": float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
            "max_dd": float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
            "sharpe": float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
            "win_rate": float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
            "profit_factor": float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
            "expectancy": float(stats["Expectancy"]) if stats.get("Expectancy") is not None else None,
            "trades": int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
        }
    except Exception as e:
        return {
            "n1": n1, "atr_period": atr_period,
            "trail_mult": trail_mult,
            "use_vwap": use_vwap,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "return_pct": None, "max_dd": None, "sharpe": None,
            "win_rate": None, "profit_factor": None,
            "expectancy": None, "trades": None,
        }


In [ ]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M1.csv").expanduser()
data = load_data(file_path, verbose=True)
data = data[data.index >= data.index[-1] - pd.Timedelta(days=7)]

portfolio, stats, signals = run_backtest(data, trail_mult=2.0)
print(stats)

trades = portfolio.trades.records_readable
print(trades)

long_exits, short_exits = build_chandelier_exits(data, signals, 2.0)

free(data, signals)



[loader] cache hit: XAUUSD_M1__cache_v1.parquet
Start                                                 0
End                                                5495
Period                                  3 days 19:36:00
Start Value                                     50000.0
End Value                                  49984.371325
Total Return [%]                              -0.031257
Benchmark Return [%]                          -3.831117
Max Gross Exposure [%]                         0.200808
Total Fees Paid                               16.874613
Max Drawdown [%]                                0.03191
Max Drawdown Duration                   3 days 18:35:00
Total Trades                                        563
Total Closed Trades                                 562
Total Open Trades                                     1
Open Trade PnL                                   -0.015
Win Rate [%]                                  52.491103
Best Trade [%]                                 0.414758


In [ ]:
portfolio.plot()

## Optimization

In [2]:

from itertools import product
import time


db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    smooth_period INTEGER NOT NULL,
    trail_mult REAL NOT NULL,
    use_vwap INTEGER NOT NULL,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL
)
""")
conn.commit()

grid = list(product(
    ["M1", "M5", "M15"], # timeframes
    range(5, 15, 2), # fast ema
    range(6, 16, 2), # smoothing
    [14, 18, 21], # atr period
    [1.0, 2.0, 3.0], # trail mult
    [False, True], # use vwap
))

#* Only for test
# grid = list(product(
#     ["M1"], # timeframes
#     [9], # fast ema
#     [12], # smoothing
#     [14], # atr period
#     [2.0], # trail mult
#     [True, False], # use vwap
# ))

t0 = time.time()

print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
asset = "XAUUSD"
# Load each timeframe once and reuse across all param combos (big memory win on M1).
data_by_tf: dict[tuple[str, bool], pd.DataFrame] = {}
for tf in sorted({g[0] for g in grid}):
    for uv in sorted({g[5] for g in grid if g[0] == tf}):
        fp = Path(f"~/market_data/{asset}/{asset}_{tf}.csv").expanduser()
        data_by_tf[(tf, uv)] = load_data(fp, verbose=True, use_vwap=uv)

ONE_LOOP_MEAN_TIME = 0.0
BATCH_SIZE = 50
for i, (timeframe, n1, smoothing, ap, tm, uv) in enumerate(grid):

    data = data_by_tf[(timeframe, uv)]

    t1 = time.time()
    result = run_params(data, n1, ap, trail_mult=tm, use_vwap=uv, asset=asset, timeframe=timeframe, smooth_period=smoothing)
    elapsed = time.time() - t1
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, atr_period, smooth_period, trail_mult, use_vwap, asset, timeframe, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["atr_period"], result["smooth_period"], result["trail_mult"], result["use_vwap"], result["asset"], result["timeframe"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    if (i + 1) % BATCH_SIZE == 0:
        conn.commit()
    ONE_LOOP_MEAN_TIME = (elapsed + ONE_LOOP_MEAN_TIME * i) / (i + 1) if i else elapsed
    combinations_left = len(grid) - (i + 1)
    est_left_sec = ONE_LOOP_MEAN_TIME * combinations_left
    if (i + 1) % max(1, len(grid) // 100) == 0 or (i + 1) == len(grid):
        print(
            f"[{time.strftime('%H:%M:%S')}] {i+1:>5}/{len(grid)} tf={timeframe} n1={n1} ap={ap} tm={tm} vwap={uv} "
            f"| sharpe={result['sharpe']} ret={result['return_pct']} trades={result['trades']} ({elapsed:.1f}s) "
            f"| mean={ONE_LOOP_MEAN_TIME:.1f}s est_left={est_left_sec/60:.1f}min"
        )

    free()

conn.commit()
free(data_by_tf)

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["sharpe_ratio"])
sweep_df = sweep_df.sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null: {len(sweep_df)}")
sweep_df.head(20)



[09:57:07] Testing 1350 combinations -> f0_smoothing_chandelier_exit
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[09:57:56]    13/1350 tf=M1 n1=5 ap=21 tm=1.0 vwap=False | sharpe=-157.46670834554283 ret=-24.55593593872827 trades=422836 (3.4s) | mean=3.5s est_left=78.6min
[09:58:35]    26/1350 tf=M1 n1=5 ap=18 tm=1.0 vwap=True | sharpe=-68.5358050990747 ret=-8.291402689747516 trades=140788 (2.0s) | mean=3.2s est_left=69.8min
[09:59:15]    39/1350 tf=M1 n1=5 ap=14 tm=2.0 vwap=False | sharpe=-96.36469501630393 ret=-15.563404048684637 trades=273014 (3.2s) | mean=3.1s est_left=66.8min
[09:59:54]    52/1350 tf=M1 n1=5 ap=21 tm=2.0 vwap=True | sharpe=-39.35818906397399 ret=-5.258270502207053 trades=90139 (2.5s) | mean=3.0s est_left=65.1min
[10:00:33] 

,id,n1,atr_period,smooth_period,trail_mult,use_vwap,asset,timeframe,expectancy,win_rate,max_dd,sharpe_ratio,profit_factor,num_trades,return_pct
0,972,5,21,12,3.0,1,XAUUSD,M15,-0.016216,32.634458,0.195525,-1.395211,0.881553,5486,-0.176554
1,966,5,18,12,3.0,1,XAUUSD,M15,-0.017272,32.489146,0.210382,-1.494823,0.873515,5529,-0.189630
2,990,5,21,14,3.0,1,XAUUSD,M15,-0.017688,32.204001,0.211327,-1.563518,0.869145,5550,-0.194970
3,1188,11,21,6,3.0,1,XAUUSD,M15,-0.017951,32.024062,0.217430,-1.589907,0.868372,5653,-0.201587
4,1044,7,21,10,3.0,1,XAUUSD,M15,-0.018291,31.883806,0.226105,-1.637843,0.865633,5750,-0.209797
5,1278,13,21,6,3.0,1,XAUUSD,M15,-0.018114,32.076801,0.228896,-1.638947,0.865650,5678,-0.204920
6,948,5,18,10,3.0,1,XAUUSD,M15,-0.019317,31.876934,0.223424,-1.642220,0.861854,5494,-0.210886
7,1116,9,21,8,3.0,1,XAUUSD,M15,-0.018408,31.793807,0.227778,-1.657106,0.864659,5782,-0.212318
8,1062,7,21,12,3.0,1,XAUUSD,M15,-0.018190,31.633715,0.229533,-1.663544,0.865081,5767,-0.209017
9,1098,9,21,6,3.0,1,XAUUSD,M15,-0.019604,31.697708,0.229375,-1.692774,0.859133,5585,-0.217609


In [14]:
vwap_comparison = pd.read_sql_query(
    f"""SELECT use_vwap,
              COUNT(*) AS n,
              AVG(expectancy) AS mean_expectancy,
              AVG(win_rate) AS mean_win_rate,
              AVG(max_dd) AS mean_max_dd,
              AVG(sharpe_ratio) AS mean_sharpe,
              AVG(profit_factor) AS mean_profit_factor,
              AVG(num_trades) AS mean_num_trades,
              AVG(return_pct) AS mean_return_pct
       FROM {TABLE_NAME}
       WHERE sharpe_ratio IS NOT NULL
       GROUP BY use_vwap""",
    conn,
)
print(vwap_comparison)


   use_vwap     n  mean_expectancy  mean_win_rate  mean_max_dd  mean_sharpe  \
0         0  2025        -0.029157      47.971807     6.034179   -37.868356   
1         1  2025        -0.026545      30.427113     1.898985   -15.206586   

   mean_profit_factor  mean_num_trades  mean_return_pct  
0            0.569467    104702.850864        -6.032046  
1            0.638994     32874.030123        -1.889997  


In [ ]:
n1_comparison = pd.read_sql_query(
    """
    SELECT n1,
           COUNT(*) AS n,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE sharpe_ratio IS NOT NULL
    GROUP BY n1
    ORDER BY n1
    """,
    conn,
)
print(n1_comparison)

In [ ]:
db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)


n1_comparison_m5 = pd.read_sql_query(
    """
    SELECT n1,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE sharpe_ratio IS NOT NULL AND timeframe like 'M5'
    GROUP BY n1
    ORDER BY mean_win_rate, mean_expectancy
    """,
    conn,
)
print(n1_comparison_m5)